In [3]:
import kagglehub
import pandas as pd
import numpy as np
import os
import unicodedata
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
from collections import Counter


## Descarga de datos con Kaggle

In [ ]:
path = kagglehub.dataset_download("bandrehc/lima-restaurant-review")

print("Path to dataset files:", path)

archivo1= os.path.join(path, 'Lima_Restaurants_2025_08_13.csv')

reviews = pd.read_csv(archivo1)

archivo2= os.path.join(path, 'restaurant_metadata.csv')

places= pd.read_csv(archivo2)

df = reviews.merge(
    places,
    on="id_place",
    how="left"
)

df.head()

Path to dataset files: C:\Users\Miguel\.cache\kagglehub\datasets\bandrehc\lima-restaurant-review\versions\2


,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,url_user,url_place,id_place
0,ChZDSUhNMG9nS0VJQ0FnSUNVLV9HNEJ3EAE,NaN,Hace 6 años,2019-08-14 02:55:32.748162,2025-08-12 02:55:32.748162,5,Eduardo Rentería,0,https://www.google.com/maps/contrib/1146375908...,https://www.google.com/maps/place/?q=place_id:...,ChIJ--Rcshq4BZERPIDegz6p5-o
1,ChZDSUhNMG9nS0VJQ0FnSURrb2FUZk5REAE,NaN,Hace 6 años,2019-08-14 02:55:32.748573,2025-08-12 02:55:32.748573,5,Margot Menacho,0,https://www.google.com/maps/contrib/1152404922...,https://www.google.com/maps/place/?q=place_id:...,ChIJ--Rcshq4BZERPIDegz6p5-o
2,ChdDSUhNMG9nS0VJQ0FnSUNFdEtlbW93RRAB,NaN,Hace 6 años,2019-08-14 02:55:32.749164,2025-08-12 02:55:32.749164,5,Johrdan Davila Ruiz,2,https://www.google.com/maps/contrib/1146353983...,https://www.google.com/maps/place/?q=place_id:...,ChIJ--Rcshq4BZERPIDegz6p5-o
3,ChdDSUhNMG9nS0VJQ0FnSUNFNE5PYV9BRRAB,"Sirven menú variado de lunes a viernes, en bas...",Hace 6 años,2019-08-14 02:55:32.749667,2025-08-12 02:55:32.749667,3,Luis Anticona,31,https://www.google.com/maps/contrib/1029463912...,https://www.google.com/maps/place/?q=place_id:...,ChIJ--Rcshq4BZERPIDegz6p5-o
4,ChZDSUhNMG9nS0VJQ0FnSURZNzV6akRREAE,Todo muy rico peeo caro,Hace 6 años,2019-08-14 02:55:32.750200,2025-08-12 02:55:32.750200,4,Adrian Rebaza,67,https://www.google.com/maps/contrib/1063973783...,https://www.google.com/maps/place/?q=place_id:...,ChIJ--Rcshq4BZERPIDegz6p5-o


,id_place,url_place,title,category,address,phoneNumber,completePhoneNumber,domain,url,stars,reviews,district,lat,long
0,ChIJ41RbR-W3BZERtC40lE27kJI,https://www.google.com/maps/place/?q=place_id:...,La 73,Restaurante,"Av. el Sol 175, Barranco 15063",(01) 2470780,+51 1 2470780,instagram.com,https://instagram.com/la73_barranco?utm_medium...,4.4,1741.0,Barranco,-12.139661,-77.023216
1,ChIJq6DzbfK3BZERzmnElUR7PUM,https://www.google.com/maps/place/?q=place_id:...,Pan Sal Aire,Restaurante,"Av. Almte. Miguel Grau 320, , Lima, Barranco 1...",951 118 993,+51 951 118 993,delivery.pansalaire.pe,https://delivery.pansalaire.pe/,4.6,2229.0,Barranco,-12.148280,-77.021087
2,ChIJQwRnc-23BZERpyhQvLJVMTQ,https://www.google.com/maps/place/?q=place_id:...,Restaurante Javier,Restaurante peruano,"Bajada de Baños 408, Barranco 15063",984 799 808,+51 984 799 808,www.restaurantejavier.pe,https://www.restaurantejavier.pe/,4.3,6810.0,Barranco,-12.149831,-77.023228
3,ChIJbYWvNb63BZER3Mb9WXTrobo,https://www.google.com/maps/place/?q=place_id:...,Nuestro Bistro,Restaurante,"Jr. Colina 107, Barranco 15063",948 721 346,+51 948 721 346,www.instagram.com,https://www.instagram.com/nuestro_bistro/,4.9,122.0,Barranco,-12.145981,-77.022046
4,ChIJQWG-5e63BZERioL90gT0vZ8,https://www.google.com/maps/place/?q=place_id:...,Cala Restaurante & Lounge,Restaurante,"Circuito de Playas, Barranco 15063",998 247 326,+51 998 247 326,NaN,NaN,4.4,8900.0,Barranco,-12.144648,-77.025753


## Cruce de datos

In [7]:
df = reviews.merge(
    places,
    on="id_place",
    how="left"
)

df.head()

,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,url_user,url_place_x,...,address,phoneNumber,completePhoneNumber,domain,url,stars,reviews,district,lat,long
0,ChZDSUhNMG9nS0VJQ0FnSUNVLV9HNEJ3EAE,NaN,Hace 6 años,2019-08-14 02:55:32.748162,2025-08-12 02:55:32.748162,5,Eduardo Rentería,0,https://www.google.com/maps/contrib/1146375908...,https://www.google.com/maps/place/?q=place_id:...,...,"Doña Virginia 105, Santiago de Surco 15049",(01) 7730584,+51 1 7730584,NaN,NaN,4.3,29.0,Surco,-12.140145,-76.999662
1,ChZDSUhNMG9nS0VJQ0FnSURrb2FUZk5REAE,NaN,Hace 6 años,2019-08-14 02:55:32.748573,2025-08-12 02:55:32.748573,5,Margot Menacho,0,https://www.google.com/maps/contrib/1152404922...,https://www.google.com/maps/place/?q=place_id:...,...,"Doña Virginia 105, Santiago de Surco 15049",(01) 7730584,+51 1 7730584,NaN,NaN,4.3,29.0,Surco,-12.140145,-76.999662
2,ChdDSUhNMG9nS0VJQ0FnSUNFdEtlbW93RRAB,NaN,Hace 6 años,2019-08-14 02:55:32.749164,2025-08-12 02:55:32.749164,5,Johrdan Davila Ruiz,2,https://www.google.com/maps/contrib/1146353983...,https://www.google.com/maps/place/?q=place_id:...,...,"Doña Virginia 105, Santiago de Surco 15049",(01) 7730584,+51 1 7730584,NaN,NaN,4.3,29.0,Surco,-12.140145,-76.999662
3,ChdDSUhNMG9nS0VJQ0FnSUNFNE5PYV9BRRAB,"Sirven menú variado de lunes a viernes, en bas...",Hace 6 años,2019-08-14 02:55:32.749667,2025-08-12 02:55:32.749667,3,Luis Anticona,31,https://www.google.com/maps/contrib/1029463912...,https://www.google.com/maps/place/?q=place_id:...,...,"Doña Virginia 105, Santiago de Surco 15049",(01) 7730584,+51 1 7730584,NaN,NaN,4.3,29.0,Surco,-12.140145,-76.999662
4,ChZDSUhNMG9nS0VJQ0FnSURZNzV6akRREAE,Todo muy rico peeo caro,Hace 6 años,2019-08-14 02:55:32.750200,2025-08-12 02:55:32.750200,4,Adrian Rebaza,67,https://www.google.com/maps/contrib/1063973783...,https://www.google.com/maps/place/?q=place_id:...,...,"Doña Virginia 105, Santiago de Surco 15049",(01) 7730584,+51 1 7730584,NaN,NaN,4.3,29.0,Surco,-12.140145,-76.999662


In [9]:
df["caption"]

0                                                       NaN
1                                                       NaN
2                                                       NaN
3         Sirven menú variado de lunes a viernes, en bas...
4                                   Todo muy rico peeo caro
                                ...                        
378964    Atención rápida y el sabor del chaufa me fue m...
378965    Pedi aeropuerto taypa con sopa. La sopa estaba...
378966                                                  NaN
378967    Buena sazon , atencion rapida y comoda . Preci...
378968    Chifa YUE HAI. Calle Inca 906, esquina con Dom...
Name: caption, Length: 378969, dtype: object

# Procesamiento de Lenguaje Natural

In [10]:
def quitar_tildes(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore")
    texto = texto.decode("utf-8")
    return texto

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = quitar_tildes(texto)
    # eliminar urls
    texto = re.sub(r"http\S+|www\S+", " ", texto)
    # eliminar números
    texto = re.sub(r"\d+", " ", texto)
    # eliminar signos y caracteres raros
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    # normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["texto_limpio"] = df["caption"].apply(limpiar_texto)

df[["caption", "texto_limpio"]].head()

,caption,texto_limpio
0,NaN,
1,NaN,
2,NaN,
3,"Sirven menú variado de lunes a viernes, en bas...",sirven menu variado de lunes a viernes en base...
4,Todo muy rico peeo caro,todo muy rico peeo caro


In [11]:
nltk.download("stopwords")

stopwords_es= set(stopwords.words("spanish"))
# agregamos palabras comunes que no aportan mucho
stopwords_extra = {
    "restaurante", "lugar", "comida", "servicio", "si", "mas", "muy",
    "solo", "bien", "mal", "vez", "tambien", "tan", "hace", "hacer"
}

stopwords_total = stopwords_es.union(stopwords_extra)

def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_total and len(t) > 2]
    return " ".join(tokens)

df["texto_procesado"] = df["texto_limpio"].apply(remover_stopwords)

df[["caption", "texto_procesado"]].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Miguel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,caption,texto_procesado
0,NaN,
1,NaN,
2,NaN,
3,"Sirven menú variado de lunes a viernes, en bas...",sirven menu variado lunes viernes base criolla...
4,Todo muy rico peeo caro,rico peeo caro


In [15]:
palabras_positivas = {
    # Calidad general
    "bueno", "buena", "buenos", "buenas",
    "excelente", "perfecto", "perfecta", "genial",
    "increible", "maravilloso", "maravillosa",
    "agradable", "recomendado", "recomendada", "recomiendo",
    "favorito", "favorita", "calidad",

    # Comida
    "rico", "rica", "ricos", "ricas",
    "delicioso", "deliciosa", "deliciosos", "deliciosas",
    "sabroso", "sabrosa", "sabrosos", "sabrosas",
    "fresco", "fresca", "frescos", "frescas",
    "jugoso", "jugosa", "crocrante", "crocante",
    "caliente", "suave", "contundente",
    "exquisito", "exquisita", "espectacular",

    # Atención
    "amable", "amables", "atento", "atenta", "atentos", "atentas",
    "cordial", "servicial", "rapido", "rapida", "rapidos", "rapidas",
    "eficiente", "paciente", "educado", "educada",

    # Ambiente
    "limpio", "limpia", "limpios", "limpias",
    "comodo", "comoda", "comodos", "comodas",
    "bonito", "bonita", "bonitos", "bonitas",
    "tranquilo", "tranquila", "acogedor", "acogedora",
    "ordenado", "ordenada",

    # Precio/valor
    "barato", "barata", "economico", "economica",
    "justo", "justa", "accesible", "promocion"
}


palabras_negativas = {
    # Calidad general
    "malo", "mala", "malos", "malas",
    "pesimo", "pesima", "horrible", "terrible",
    "decepcion", "decepcionante", "fatal",
    "deficiente", "regular", "mediocre",

    # Comida
    "frio", "fria", "frios", "frias",
    "quemado", "quemada", "quemados", "quemadas",
    "crudo", "cruda", "crudos", "crudas",
    "salado", "salada", "salados", "saladas",
    "insipido", "insipida", "duro", "dura",
    "grasoso", "grasosa", "seco", "seca",
    "feo", "fea", "malogrado", "malograda",
    "pequeño", "pequena", "escaso", "escasa",

    # Atención
    "lento", "lenta", "lentos", "lentas",
    "demora", "demoro", "demoraron", "demorado", "demorada",
    "tardo", "tardaron", "espera", "cola",
    "grosero", "grosera", "malcriado", "malcriada",
    "desatento", "desatenta", "indiferente",

    # Ambiente
    "sucio", "sucia", "sucios", "sucias",
    "ruidoso", "ruidosa", "incomodo", "incomoda",
    "desordenado", "desordenada", "oscuro", "oscura",
    "caluroso", "calurosa",

    # Precio
    "caro", "cara", "caros", "caras",
    "costoso", "costosa", "excesivo", "excesiva",
    "sobrevalorado", "sobrevalorada",

    # Delivery / pedido
    "incompleto", "incompleta", "derramado", "derramada",
    "roto", "rota", "equivocado", "equivocada",
    "reclamo", "queja", "problema", "problemas"
}


negaciones = {
    "no", "nunca", "jamas", "tampoco", "ni"
}


intensificadores = {
    "muy", "demasiado", "super", "bastante", "tan", "re", "sumamente"
}

In [16]:
def analizar_sentimiento_diccionario(texto):
    texto_limpio = limpiar_texto(texto)
    tokens = texto_limpio.split()
    
    if len(tokens) == 0:
        return pd.Series({
            "sentimiento_pln": "sin_texto",
            "score_sentimiento": 0,
            "positivas_detectadas": "",
            "negativas_detectadas": ""
        })
    
    score = 0
    positivas_detectadas = []
    negativas_detectadas = []
    
    for i, token in enumerate(tokens):
        valor = 0
        
        if token in palabras_positivas:
            valor = 1
            positivas_detectadas.append(token)
        
        elif token in palabras_negativas:
            valor = -1
            negativas_detectadas.append(token)
        
        # Revisar negaciones hasta 3 palabras antes
        ventana_anterior = tokens[max(0, i-3):i]
        
        if valor != 0 and any(neg in ventana_anterior for neg in negaciones):
            valor = valor * -1
        
        # Revisar intensificadores hasta 3 palabras antes
        if valor != 0 and any(intens in ventana_anterior for intens in intensificadores):
            valor = valor * 1.5
        
        score += valor
    
    n_pos = len(positivas_detectadas)
    n_neg = len(negativas_detectadas)
    
    # Clasificación final
    if n_pos > 0 and n_neg > 0:
        if abs(score) <= 1:
            sentimiento = "mixto"
        elif score > 1:
            sentimiento = "positivo"
        else:
            sentimiento = "negativo"
    else:
        if score > 0:
            sentimiento = "positivo"
        elif score < 0:
            sentimiento = "negativo"
        else:
            sentimiento = "neutro"
    
    return pd.Series({
        "sentimiento_pln": sentimiento,
        "score_sentimiento": score,
        "positivas_detectadas": ", ".join(sorted(set(positivas_detectadas))),
        "negativas_detectadas": ", ".join(sorted(set(negativas_detectadas)))
    })

In [17]:
comentarios_prueba = [
    "La comida estuvo deliciosa y la atención fue excelente",
    "Demoraron demasiado y la comida llegó fría",
    "El local es bonito pero la atención fue lenta",
    "No estuvo malo, la comida fue aceptable",
    "El precio es caro y el servicio pésimo",
    "Buen ambiente, buena comida y personal amable"
]

for comentario in comentarios_prueba:
    print(comentario)
    print(analizar_sentimiento_diccionario(comentario))
    print("-" * 80)

La comida estuvo deliciosa y la atención fue excelente
sentimiento_pln                     positivo
score_sentimiento                          2
positivas_detectadas    deliciosa, excelente
negativas_detectadas                        
dtype: object
--------------------------------------------------------------------------------
Demoraron demasiado y la comida llegó fría
sentimiento_pln                negativo
score_sentimiento                    -2
positivas_detectadas                   
negativas_detectadas    demoraron, fria
dtype: object
--------------------------------------------------------------------------------
El local es bonito pero la atención fue lenta
sentimiento_pln          mixto
score_sentimiento            0
positivas_detectadas    bonito
negativas_detectadas     lenta
dtype: object
--------------------------------------------------------------------------------
No estuvo malo, la comida fue aceptable
sentimiento_pln         positivo
score_sentimiento              1
p

In [18]:
col_texto = "caption"  

df["texto_limpio"] = df[col_texto].apply(limpiar_texto)

df_resultados_sentimiento = df[col_texto].apply(analizar_sentimiento_diccionario)

df = pd.concat([df, df_resultados_sentimiento], axis=1)

df[[col_texto, "sentimiento_pln", "score_sentimiento", 
    "positivas_detectadas", "negativas_detectadas"]].head()

,caption,sentimiento_pln,score_sentimiento,positivas_detectadas,negativas_detectadas
0,NaN,sin_texto,0.0,,
1,NaN,sin_texto,0.0,,
2,NaN,sin_texto,0.0,,
3,"Sirven menú variado de lunes a viernes, en bas...",positivo,1.5,acogedor,
4,Todo muy rico peeo caro,mixto,0.0,rico,caro
